# VideoDB Understanding: Robo Activity

Generate a timeline of robot actions and goals with the `robo_activity` analyzer. Understanding handles frame sampling, contact sheets and both inference passes; you choose the Gemini model and inference mode.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/robo-activity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install dependencies

In [ ]:
%pip install -q "videodb>=0.5.0" python-dotenv pandas yt-dlp

## 2. Connect to VideoDB

Set `VIDEO_DB_BASE_URL` in your environment or `.env` to a VideoDB Server or Runtime endpoint with `robo_activity` support deployed. The connected Understanding deployment must have Gemini configured.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
api_key = os.getenv("VIDEO_DB_API_KEY") or getpass("Enter your VideoDB API key: ")

conn = connect(api_key=api_key, base_url=os.environ["VIDEO_DB_BASE_URL"])
collection = conn.get_collection()

## 3. Choose a video

Download the sample video locally with `yt-dlp` and upload the file. Replace the URL with your own video, or reuse an uploaded video. Merging the video and audio streams needs `ffmpeg`, which Colab already has.

In [ ]:
!yt-dlp -q -f "bv*[ext=mp4][height<=1080]+ba[ext=m4a]/b[ext=mp4]/b" --merge-output-format mp4 -o laundry-folding-pov.mp4 "https://www.youtube.com/watch?v=wuRf25_zME0"

In [ ]:
video = collection.upload(file_path="laundry-folding-pov.mp4")

# Alternatively, skip the download cell and replace the file upload above with:
# video = collection.upload(url="https://www.youtube.com/watch?v=wuRf25_zME0")

# Or reuse an uploaded video:
# video = collection.get_video("YOUR_VIDEO_ID")

video.play()

## 4. Run Robo Activity

Choose a Gemini model available to your deployment and set `MODE`:

- `batch`: submit Gemini batch jobs for both passes. This is the default mode.
- `inline`: make direct Gemini requests for both passes.

The Understanding job is asynchronous in either mode. Sampling, contact sheets, prompts and pass dependencies are managed by the analyzer; no custom pipeline configuration is needed.

In [ ]:
MODEL = "google/gemini-2.5-pro"
MODE = "batch"  # Change to "inline" for direct Gemini requests.

robo = video.understand(
    analyzers=[
        {
            "type": "robo_activity",
            "name": "robo",
            "config": {"model": MODEL, "mode": MODE},
        }
    ],
)

print("Understanding ID:", robo.id)

In [ ]:
# Both passes can submit batch jobs, so allow a long wait.
# On a timeout, rerun this cell to keep waiting for the same run.
robo.wait_until_complete(timeout=48 * 3600, poll_interval=60)
analyzer = robo.get_analyzer("robo", refresh=True)
print("Status:", analyzer.status)
if not analyzer.is_successful:
    raise RuntimeError(f"Robo Activity did not succeed: {analyzer.status}")

## 5. View actions and goals

Each scene contains the action's start/end times and data such as its instruction, goal and boundary details. Top-level metadata contains the goals, pieces and processing warnings.

In [ ]:
import pandas as pd

robo_output = analyzer.get_output()
pd.json_normalize(robo_output["scenes"]).head()

In [ ]:
pd.DataFrame(robo_output["metadata"]["goals"])